# Lecture 9: Annotated Text Corpora I

So far in this course, we've been working with annotated data without really
naming it as such. When we used `brown.tagged_words()` or
`nltk.pos_tag()`, we were accessing **linguistic annotations** — structured
information that's been layered on top of raw text.

Today we'll step back and think about annotation more broadly:
- What does it mean to annotate text?
- How are syntactic structures represented in corpora?
- How do we work with annotated treebank data in Python?

**Prerequisites:**
- Python and the `ling250` environment
- Familiarity with POS tagging and tagged corpora from Lectures 5-6

---

## Part 1: What is Linguistic Annotation?

Raw text is just a sequence of characters. To do linguistic analysis, we
often need to add layers of structured information on top of that text.
This is **annotation** — the process of marking up text with linguistic
categories and relationships.

You've already worked with one level of annotation: **Part-of-Speech
tagging**, where each word gets labeled with its grammatical category.
But POS tags are just one layer. Annotation can happen at many levels:

| Level | What's annotated | Example |
|-------|------------------|---------|
| Tokenization | Word boundaries | "don't" → "do" + "n't" |
| POS tagging | Grammatical category per word | dog/NN, chased/VBD |
| Syntax | Phrase structure or dependencies | [The dog] chased [the cat] |
| Named entities | People, places, organizations | [Barack Obama]/PERSON |
| Semantics | Word senses, semantic roles | "bank" = financial institution |
| Discourse | Coreference, rhetorical structure | "he" refers to "the dog" |

Each level builds on the ones below it — you generally need tokenization
before POS tagging, and POS tagging before parsing.

### Who annotates?

Annotated corpora are created through some combination of:
- **Trained human annotators** following detailed guidelines (most reliable,
  most expensive)
- **Crowdsourcing** (e.g., Amazon Mechanical Turk) with quality controls
- **Automatic tools** with human correction — a model tags the text first,
  then humans review and fix mistakes

The important thing to understand is that annotation involves
**interpretation**. Annotators don't just record objective facts — they
make judgment calls. Is "running" a verb or an adjective in "running
water"? Is "New York Times" one entity or three words? Reasonable people
can disagree.

This is why annotation projects measure **inter-annotator agreement
(IAA)**: if two annotators independently label the same data, how often
do they make the same choice? High agreement means the annotation scheme
is well-defined; low agreement often signals genuine ambiguity in the
language.

---

## Part 2: Tagsets Compared

We've already used POS tags, but we haven't thought much about the fact that
different corpora use **different tagsets**. The Brown Corpus, the Penn
Treebank, and the Universal tagset all label the same grammatical
distinctions differently — and with different levels of detail.

Let's look at how the same data gets tagged under each system.

In [ ]:
import nltk
from nltk.corpus import brown, treebank

In [ ]:
# The Brown Corpus has its own tagset with ~470 unique tags
brown_sent = brown.tagged_sents()[0]
print("Brown tagset:")
print(brown_sent[:8])

In [ ]:
# The same sentence with the Universal tagset (12 tags)
brown_sent_uni = brown.tagged_sents(tagset='universal')[0]
print("Universal tagset:")
print(brown_sent_uni[:8])

In [ ]:
# The Penn Treebank uses ~45 tags
ptb_sent = treebank.tagged_sents()[0]
print("Penn Treebank tagset:")
print(ptb_sent[:8])

### Why so many tagsets?

| Tagset | # Tags | Granularity | Example: determiner |
|--------|--------|-------------|---------------------|
| Brown | ~470 | Very fine | AT, AT-TL, DTI, DTS, ... |
| Penn Treebank | ~45 | Medium | DT, WDT, PDT |
| Universal | 12 | Coarse | DET |

There's a fundamental tradeoff in tagset design:
- **Fine-grained tags** capture more distinctions, but they require more
  annotator training, are harder to apply consistently, and tend to be
  specific to one language
- **Coarse tags** are easier to use, make it possible to compare across
  languages, but lose information that might matter for some analyses

The Brown Corpus (1960s) used extremely detailed tags because the goal
was to capture every distinction that might matter for any future research.
The Penn Treebank (1990s) simplified this to a more practical set. The
Universal tagset (2012) went further, aiming for tags that work across
all languages.

Which tagset you use depends on your research question. If you're studying
the distribution of verb tenses in English, you want fine-grained tags
that distinguish VB, VBD, VBG, VBN, VBP, VBZ. If you're comparing noun
usage across English and Japanese, you want a tagset that both languages
share.

### POS ambiguity

One reason annotation requires human judgment: many words can belong to
multiple categories depending on context. The same word gets different tags
in different sentences.

In [ ]:
# "run" can be a noun or a verb
print(nltk.pos_tag(['I', 'run', 'every', 'day']))
print(nltk.pos_tag(['That', 'was', 'a', 'good', 'run']))

In [ ]:
# "that" can be a determiner, complementizer, or relative pronoun
print(nltk.pos_tag(['That', 'dog', 'is', 'cute']))       # determiner
print(nltk.pos_tag(['I', 'know', 'that', 'she', 'left'])) # complementizer
print(nltk.pos_tag(['The', 'book', 'that', 'I', 'read'])) # relative pronoun

When human annotators disagree on cases like these, that's not necessarily a
mistake — it can reflect genuine ambiguity in language. This is exactly
what inter-annotator agreement tries to measure.

---

## Part 3: Constituency

POS tags annotate individual words, but linguists also care about how words
group together into larger units. Consider the sentence:

> The big dog chased the cat.

"The big dog" acts as a unit here — it's the thing doing the chasing. You
could replace it with a single pronoun ("It chased the cat") or swap in
another phrase of the same type ("My neighbor chased the cat"). A group of
words that acts as a unit in this way is called a **constituent**.

A **Noun Phrase (NP)** is a constituent that acts like a noun — or, you
can think of it as an extension of a noun. "dog" is a noun; "the big dog"
is a noun phrase built around that noun. Similarly:
- A **Verb Phrase (VP)** is built around a verb: "chased the cat"
- A **Prepositional Phrase (PP)** is built around a preposition: "on the mat"
- **S** (Sentence) is the top-level constituent

A **constituency tree** (or **parse tree**) represents this hierarchical
grouping — which words combine into phrases, and which phrases combine
into larger phrases.

### Context-Free Grammars

The formal system behind constituency trees is called a **Context-Free
Grammar (CFG)**. A CFG is a set of rules that describe how phrases can
be built. For example:

- `S → NP VP` — a sentence can be a noun phrase followed by a verb phrase
- `NP → DT NN` — a noun phrase can be a determiner followed by a noun
- `NP → DT JJ NN` — or a determiner, adjective, and noun
- `VP → VBD NP` — a verb phrase can be a past-tense verb followed by an NP
- `PP → IN NP` — a prepositional phrase is a preposition followed by an NP

**Parsing** is the process of using rules like these to produce a tree
for a given sentence. But for our purposes, parsing is less important
than the **output** — a parsed sentence is an **annotation of syntactic
structure**. Someone (or some system) has analyzed how the words group
together, and recorded that analysis as a tree.

The **Penn Treebank** stores these trees in a text format called
**bracket notation**:

In [ ]:
from nltk import Tree

# A simple sentence in bracket notation
bracket_string = "(S (NP (DT The) (NN dog)) (VP (VBD chased) (NP (DT the) (NN cat))))"

tree = Tree.fromstring(bracket_string)
tree.pretty_print()

Reading bracket notation:
- Each `(LABEL ...)` defines a node in the tree
- Leaf nodes are words with their POS tag: `(NN dog)` means "dog" is a noun
- Internal nodes group their children into a phrase:
  `(NP (DT The) (NN dog))` means "The dog" is a noun phrase
- `S` at the top represents the whole sentence

We can also extract the CFG rules that the tree embodies:

In [ ]:
for production in tree.productions():
    print(production)

---

## Part 4: Working with NLTK Trees

NLTK's `Tree` class lets us navigate and search constituency trees
programmatically. This is what makes treebanks useful as data — we can
ask questions about syntactic structure across thousands of sentences.

### Basic navigation

In [ ]:
# The label of the root node
print("Root label:", tree.label())

# The leaves (words) of the tree
print("Leaves:", tree.leaves())

# The height of the tree
print("Height:", tree.height())

In [ ]:
# A Tree acts like a list of its children
print("Number of children:", len(tree))
print()

# First child (the NP)
print("First child:")
print(tree[0])
print()

# Second child (the VP)
print("Second child:")
print(tree[1])

In [ ]:
# You can chain indexing to go deeper
# tree[1] is the VP, tree[1][1] is the NP inside the VP
print("The object NP:")
print(tree[1][1])
print("Its leaves:", tree[1][1].leaves())

### Finding subtrees

The `.subtrees()` method lets us find all nodes in the tree that match a
condition. This is powerful for searching treebanks — it's how we can
ask questions like "find every noun phrase" or "find every PP inside a VP".

In [ ]:
# Find all NPs in our simple tree
for subtree in tree.subtrees(filter=lambda t: t.label() == 'NP'):
    print("NP:", ' '.join(subtree.leaves()))

In [ ]:
# Try a more complex sentence
complex_str = """(S
  (NP (DT The) (JJ old) (NN man))
  (VP (VBD sat)
    (PP (IN on)
      (NP (DT the) (NN bench)))
    (PP (IN in)
      (NP (DT the) (NN park)))))"""

complex_tree = Tree.fromstring(complex_str)
complex_tree.pretty_print()

In [ ]:
# Find all NPs
print("All NPs:")
for subtree in complex_tree.subtrees(filter=lambda t: t.label() == 'NP'):
    print("  ", ' '.join(subtree.leaves()))

print()

# Find all PPs
print("All PPs:")
for subtree in complex_tree.subtrees(filter=lambda t: t.label() == 'PP'):
    print("  ", ' '.join(subtree.leaves()))

---

## Part 5: The Penn Treebank

The **Penn Treebank (PTB)** is one of the most influential annotated corpora
in the history of NLP. Created at the University of Pennsylvania in the
early 1990s, it contains about 40,000 sentences from the Wall Street
Journal, each annotated with:
- POS tags (the Penn Treebank tagset we've already used)
- Full constituency trees

The PTB was the standard benchmark for training and evaluating parsers for
over a decade. Its tagset and annotation conventions influenced nearly every
English NLP tool built since.

NLTK includes a sample of the PTB that we can explore directly.

In [ ]:
# nltk.download('treebank')  # uncomment if needed

print(f"Number of parsed sentences: {len(treebank.parsed_sents())}")

In [ ]:
# Look at a parsed sentence
ptb_tree = treebank.parsed_sents()[0]
print("Sentence:", ' '.join(ptb_tree.leaves()))
print()
ptb_tree.pretty_print()

Notice some differences from our simple examples:
- Labels like `NP-SBJ` (subject NP), `PP-CLR` (closely related PP),
  `NP-TMP` (temporal NP) — these are **function tags** that go beyond
  phrase type to indicate the phrase's grammatical **role** in the sentence
- The trees are deeper and more complex than our hand-built examples —
  real sentences have a lot of structure

In [ ]:
# A simpler sentence from the treebank
simple_ptb = treebank.parsed_sents()[76]
print("Sentence:", ' '.join(simple_ptb.leaves()))
print()
simple_ptb.pretty_print()

### Searching across a treebank

The real power of a treebank is being able to search for structural
patterns across many sentences. This is something you simply can't do
with raw text or even POS-tagged text — you need the tree structure.

Let's start with a question: **how often do English sentences begin with
a prepositional phrase?** (e.g., "In the morning, she left.")

In [ ]:
# Find sentences where the first child of S is a PP
pp_fronted = []
for parsed_sent in treebank.parsed_sents():
    # Skip sentences where the root isn't S
    if parsed_sent.label() != 'S':
        continue
    first_child = parsed_sent[0]
    if hasattr(first_child, 'label') and first_child.label().startswith('PP'):
        pp_fronted.append(parsed_sent)

print(f"Sentences starting with PP: {len(pp_fronted)} / {len(treebank.parsed_sents())}")
print()

# Show a few examples
for t in pp_fronted[:5]:
    print(' '.join(t.leaves()))
    print()

We can also ask broader quantitative questions about the corpus as a whole.

In [ ]:
# What are the most common phrase types across the whole treebank?
from nltk import FreqDist

all_labels = []
for parsed_sent in treebank.parsed_sents():
    for subtree in parsed_sent.subtrees():
        all_labels.append(subtree.label())

label_fdist = FreqDist(all_labels)
print("Most common phrase labels:")
for label, count in label_fdist.most_common(15):
    print(f"  {label:12s} {count:6d}")

In [ ]:
# How long are NPs in the treebank?
np_lengths = []
for parsed_sent in treebank.parsed_sents():
    for subtree in parsed_sent.subtrees():
        if subtree.label() == 'NP':
            np_lengths.append(len(subtree.leaves()))

np_length_fdist = FreqDist(np_lengths)

print(f"Total NPs found: {len(np_lengths)}")
print(f"Average NP length: {sum(np_lengths) / len(np_lengths):.1f} words")
print()
print("NP length distribution (most common):")
for length, count in sorted(np_length_fdist.most_common(10)):
    print(f"  {length} words: {count}")

---

## Part 6: From Trees to Research Questions

The point of treebank annotation isn't just to have pretty trees — it's to
enable us to ask **structural questions** about language that would be
impossible with raw text alone.

Some examples:
- How complex are noun phrases in news text vs. fiction?
- How often are subjects longer than objects?
- What kinds of phrases appear in certain positions (e.g., sentence-initial)?
- How deeply nested are typical sentences in a given genre?

These are the kinds of questions that linguistic annotation makes possible.
The annotation is a bridge between raw text and scientific analysis.

In [ ]:
# Example: How long are subject NPs in the Penn Treebank?
# The PTB marks subjects with the function tag NP-SBJ

subject_lengths = []

for parsed_sent in treebank.parsed_sents():
    for subtree in parsed_sent.subtrees():
        if subtree.label() == 'NP-SBJ':
            subject_lengths.append(len(subtree.leaves()))

print(f"Subject NPs found: {len(subject_lengths)}")
print(f"Average subject length: {sum(subject_lengths) / len(subject_lengths):.1f} words")
print()
print("Subject length distribution:")
subj_fdist = FreqDist(subject_lengths)
for length, count in sorted(subj_fdist.most_common(8)):
    print(f"  {length} words: {count}")

In [ ]:
# What's the deepest tree in the treebank?
max_height = 0
deepest_sent = None

for parsed_sent in treebank.parsed_sents():
    h = parsed_sent.height()
    if h > max_height:
        max_height = h
        deepest_sent = parsed_sent

print(f"Maximum tree height: {max_height}")
print(f"Sentence: {' '.join(deepest_sent.leaves()[:15])}...")
print(f"({len(deepest_sent.leaves())} words total)")

---

## Looking Ahead

Constituency trees represent structure by showing which words **group
together** into phrases. But there's another way to represent syntactic
structure: **dependency trees**, which show which words **modify or depend
on** other words.

For example, in "The big dog chased the cat":
- Constituency says: "The big dog" forms a group (NP)
- Dependency says: "big" and "The" both depend on "dog"; "dog" depends on
  "chased"

Next time, we'll look at dependency grammar and the **Universal
Dependencies** project — a massive effort to annotate text in 100+
languages with a consistent scheme. We'll also learn about **BIO tagging**,
a flexible annotation format that can represent many different tasks.

---

## Exercises

Try these on your own!

### Exercise 1: Build a tree

Write the bracket notation for the sentence "The cat sat on the mat" and
use `Tree.fromstring()` to parse and display it.

**Hint:** The structure should be `S → NP VP`, where the VP contains a verb
and a PP (prepositional phrase), and the PP contains a preposition and an NP.

In [ ]:
# Your code here


### Exercise 2: Find VPs containing PPs

Search the Penn Treebank for verb phrases (VP) that contain a prepositional
phrase (PP) as a direct child. How many are there? Print the words of 5
examples.

**Hint:** Use `.subtrees()` to find VPs, then check their children's labels.

In [ ]:
# Your code here


### Exercise 3: Tree height distribution

Compute the height of every parsed sentence in the treebank and find the
distribution. What's the most common tree height? Use `FreqDist` and
`.most_common()` to find out.

**Bonus:** Is there a relationship between sentence length (number of words)
and tree height?

In [ ]:
# Your code here


---

## Quick Reference

### NLTK Tree

| Code | What it does |
|------|-------------|
| `Tree.fromstring(s)` | Parse bracket notation into a Tree |
| `tree.pretty_print()` | Display the tree as ASCII art |
| `tree.label()` | Get the label of the root node |
| `tree.leaves()` | Get all words (leaf nodes) |
| `tree.height()` | Get the depth of the tree |
| `tree[i]` | Access the i-th child |
| `tree.subtrees()` | Iterate over all subtrees |
| `tree.subtrees(filter=f)` | Iterate over subtrees matching a condition |
| `tree.productions()` | Extract CFG rules from the tree |

### Treebank access

| Code | What it does |
|------|-------------|
| `treebank.parsed_sents()` | Get all sentences as Tree objects |
| `treebank.tagged_sents()` | Get all sentences as (word, tag) lists |
| `treebank.words()` | Get all words (flat) |

### Bracket notation

```
(S                          ← sentence
  (NP (DT The) (NN dog))   ← noun phrase: determiner + noun
  (VP (VBD chased)          ← verb phrase: verb + noun phrase
    (NP (DT the) (NN cat))))
```

### Common phrase labels

| Label | Meaning |
|-------|--------|
| S | Sentence |
| NP | Noun Phrase |
| VP | Verb Phrase |
| PP | Prepositional Phrase |
| ADJP | Adjective Phrase |
| ADVP | Adverb Phrase |
| SBAR | Subordinate clause |